# 01 · Data Acquisition

Downloads the full Hacker News dataset from Google BigQuery and saves it as local Parquet files for fast querying in subsequent notebooks.

**Prerequisites:**
1. Create a GCP project at [console.cloud.google.com](https://console.cloud.google.com) and enable the BigQuery API
2. Run `gcloud auth application-default login` in your terminal
3. Copy `.env.example` to `.env` and set `GCP_PROJECT_ID`

**BigQuery free tier:** 1 TB/month of queries. This notebook uses ~10 GB.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
from dotenv import load_dotenv
load_dotenv('../.env')

import duckdb
import pandas as pd
from src.loader import fetch_from_bigquery, db, STORIES_PARQUET, COMMENTS_PARQUET

GCP_PROJECT = os.environ['GCP_PROJECT_ID']
print(f'Using GCP project: {GCP_PROJECT}')

## Download from BigQuery

This takes 5–15 minutes and only needs to run once.

In [ ]:
fetch_from_bigquery(GCP_PROJECT)

## Profile the dataset

In [ ]:
con = db()

print('=== STORIES ===')
print(con.execute('SELECT COUNT(*) AS n_stories FROM stories').df())
print(con.execute('SELECT MIN(posted_at), MAX(posted_at) FROM stories').df())
print(con.execute('SELECT COUNT(*) FROM stories WHERE url IS NOT NULL').df())
print()
print('=== COMMENTS ===')
print(con.execute('SELECT COUNT(*) AS n_comments FROM comments').df())

In [ ]:
# Score distribution sanity check
con.execute("""
    SELECT
        MIN(score)  AS min_score,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY score) AS median_score,
        AVG(score)  AS mean_score,
        MAX(score)  AS max_score
    FROM stories
""").df()

In [ ]:
# Stories per year
per_year = con.execute("""
    SELECT
        YEAR(posted_at) AS year,
        COUNT(*) AS n_stories,
        AVG(score) AS avg_score
    FROM stories
    GROUP BY 1
    ORDER BY 1
""").df()

display(per_year)

In [ ]:
import matplotlib.pyplot as plt
sys.path.insert(0, '..')
from src.viz import set_style, bar_chart

set_style()
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(per_year['year'], per_year['n_stories'], color='#e8604c', edgecolor='white')
ax.set_title('Hacker News stories submitted per year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Stories')
plt.tight_layout()
plt.savefig('../data/fig_stories_per_year.png', dpi=150, bbox_inches='tight')
plt.show()